# 08. SQL Group By, Having & Rollup: Beginner Guide

### 📝 Universal SQL Execution Order (All SQL Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline (All 12 Clauses) ────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. QUALIFY         ➔ 8. SELECT & CASE      ➔ 9. DISTINCT (Dedup)           │
│ ➔ 10. UNION/INTERSECT➔ 11. ORDER BY (Sort)   ➔ 12. LIMIT / OFFSET (Page)     │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **08. SQL Group By, Having & Rollup**. Grouping transforms individual row vectors into aggregated dimensional buckets. This notebook covers single-column and multi-column grouping (`GROUP BY`), the critical distinction between pre-aggregation row filtering (`WHERE`) and post-aggregation group filtering (`HAVING`), and advanced analytical grouping hierarchies.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Dimensional Bucketing: `GROUP BY column_name`
- [x] 🔹 Multi-Attribute Grouping: `GROUP BY col1, col2`
- [x] 🔹 Post-Aggregation Group Filtering: `HAVING` Predicates
- [x] 🔹 Logical Execution: `WHERE` (Row Filter) vs `HAVING` (Group Filter)
- [x] 🔍 Scenario: Regional Card-Type Fraud Velocity & Merchant Threshold Audit











In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Categorical Collapsing: `GROUP BY`
- **What it does:** Groups raw rows sharing common values across designated key columns into single summary rows.
- **Syntax:** `SELECT key_col, AGG(val_col) FROM table_name GROUP BY key_col`
- **Dataset Application & Code Demonstration:** Calculates total transaction count and average spend grouped by card type.


In [2]:
%%sql
SELECT 
    card_type,
    COUNT(transaction_id) AS total_transactions,
    ROUND(AVG(transaction_amount), 2) AS avg_amount,
    ROUND(SUM(transaction_amount), 2) AS total_volume
FROM transactions
WHERE card_type IS NOT NULL
GROUP BY card_type
ORDER BY total_volume DESC;


,card_type,total_transactions,avg_amount,total_volume
0,Discover,3755,1016.84,3619952.00
1,Amex,3755,1002.94,3597545.46
2,MasterCard,3756,1002.26,3585088.05
3,Visa,3734,999.25,3524349.99


### 🔹 Multi-Attribute Grouping: `GROUP BY col1, col2`
- **What it does:** Partitions data across combinations of multiple categorical dimensions.
- **Syntax:** `SELECT col1, col2, AGG(val) FROM table_name GROUP BY col1, col2`
- **Dataset Application & Code Demonstration:** Segments transaction volume by both region and device type.


In [3]:
%%sql
SELECT 
    region,
    device_type,
    COUNT(transaction_id) AS tx_count,
    ROUND(SUM(transaction_amount), 2) AS region_device_volume
FROM transactions
WHERE region IS NOT NULL AND device_type IS NOT NULL
GROUP BY region, device_type
ORDER BY region ASC, region_device_volume DESC
LIMIT 8;


,region,device_type,tx_count,region_device_volume
0,East,ATM,23,21252.59
1,East,Desktop,22,20978.01
2,East,Mobile,18,20456.19
3,East,POS,14,8528.65
4,North,Desktop,21,22178.02
5,North,POS,19,21978.77
6,North,ATM,18,17153.85
7,North,Mobile,16,16357.67


### 🔹 Post-Aggregation Filtering: `HAVING`
- **What it does:** Filters aggregated summary groups based on the result of an aggregate function.
- **Syntax:** `SELECT col, AGG(val) FROM table GROUP BY col HAVING AGG(val) > threshold`
- **Dataset Application & Code Demonstration:** Isolates merchants with more than 10 transactions whose average transaction size exceeds $1,100.00.


In [4]:
%%sql
SELECT 
    merchant_id,
    COUNT(transaction_id) AS tx_count,
    ROUND(AVG(transaction_amount), 2) AS avg_ticket_size
FROM transactions
GROUP BY merchant_id
HAVING COUNT(transaction_id) > 10 
   AND AVG(transaction_amount) > 1100.00
ORDER BY avg_ticket_size DESC
LIMIT 5;


,merchant_id,tx_count,avg_ticket_size
0,M4178,35,1354.29
1,M5128,40,1294.98
2,M2257,23,1277.03
3,M6517,27,1262.01
4,M9452,32,1230.70


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: WHERE vs HAVING Query Optimizer Performance
- **Objective:** Demonstrate the performance and structural difference between pre-aggregation row filtering (`WHERE`) and post-aggregation metric evaluation (`HAVING`).
- **Approach:** Combine both clauses in a single optimized query pipeline.


In [5]:
%%sql
SELECT 
    region,
    COUNT(transaction_id) AS fraud_tx_count,
    ROUND(SUM(transaction_amount), 2) AS total_fraud_loss
FROM transactions
WHERE is_fraud = 1
GROUP BY region
HAVING SUM(transaction_amount) > 50000.00
ORDER BY total_fraud_loss DESC;


,region,fraud_tx_count,total_fraud_loss
0,West,408,722963.74
1,North,403,716308.85
2,South,371,658358.56
3,East,363,647146.83
